# Notebook 2. Do the posthoc analysis on the Mapper
## 0 · Pre-requisites

In this notebook, you can do Mapper PostHoc Analysis including color-coding them with target outcomes and save them as SVG files.
<br><span style="color:red">IMPORTANT NOTE: In addition to the three dataframes saved from the previous notebook, you require to get the configuration JSON file of the HTML Mapper. In order to do that, you should open the HTML Mapper, move and untangle it if needed, and click on "Help" to save the config JSON file. This file contains all information of nodes and edges.</span></br>

## 1. Environment check

Run the next cell once per session to import all functions.

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np
import kmapper as km
from sklearn.cluster import DBSCAN
from kmapper import jupyter
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import sklearn
import matplotlib.colors as mcolors
import json
import networkx as nx
import plotly.graph_objects as go
import plotly.colors as pc
import plotly.io as pio
import ast
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import Normalize
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler



## 2. Import three main dataframes

Update directories.

In [ ]:
df2 = pd.read_csv()
df_nodes = pd.read_csv()  
df_edges = pd.read_csv()
dfmapper = pd.read_csv()

## PostHoc 1. Plot the Mapper with no coloration as a SVG format

Please update the location of the config.json file in the code below.

In [ ]:
# Create an empty graph
G = nx.Graph()


with open("", "r") as file:
    node_positions = json.load(file)


# Add nodes with positions
for node in dfmapper['Nodes']:
    if node in node_positions:  #node_position is the config file from mapper
        x, y = node_positions[node]['x'], node_positions[node]['y']
        G.add_node(node, pos=(x, y))

# Add edges, avoiding duplicates
edges_added = set()  # To track added edges
for i, row in dfmapper.iterrows():
    node = row['Nodes']
    edges = row["Edges"]
    
    # Ensure edges are a proper list (convert if stored as a string)
    if isinstance(edges, str):
        edges = ast.literal_eval(edges)  # Convert from string to actual list

    for neighbor in edges:
        if (neighbor, node) not in edges_added and (node, neighbor) not in edges_added:
            G.add_edge(node, neighbor)
            edges_added.add((node, neighbor))

# Get positions for plotting
pos = nx.get_node_attributes(G, 'pos')
edge_x = []
edge_y = []

# Extract edge coordinates for plotting
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

# Create node trace
node_x = []
node_y = []
node_text = []

for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_text.append(node)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(size=5, color='white',         line=dict(
            width=2,         # Border width
            color='grey' # Border color (stroke)
        )),
    hoverinfo='none'
)

# Create edge trace
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.3, color='grey'),
    hoverinfo='none',
    mode='lines'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    autosize = False,
    width = 300,
    height = 300,
    showlegend=False,
    hovermode='closest',
    margin=dict(b=0, l=0, r=0, t=0),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='white',  # Set background color to white
    paper_bgcolor='white'  # Set paper background color to white
)

fig.show()

## PostHoc 2. Color your condition of interest (Single color - color intensity base)

In order to this, you need to add the condition column to the df2 dataframe. Simply open that file and add any target coloring column you want. I had PD, CVA, HC, and LL. Once you change the target_condition, it colors the Mapper only for that particular condition. You can play with RGB numbers to change the color.

In [ ]:
target_condition = "PD"
node_condition_counts = {}
with open(r"Y:\LabMembers\S Daneshgar\P2C\TDA\Results\3-18-2025 Sajjad\config.json", "r") as file:
    node_positions = json.load(file)

    
for _, row in df2.iterrows():
    nodes = row['node_numbers']
    condition = row['Condition']
    
    # Ensure nodes is a proper list
    if isinstance(nodes, str):
        nodes = ast.literal_eval(nodes)  # Convert from string to actual list
    
    for node in nodes:
        node = str(node)  # Ensure it's treated as a string (not an integer)
        if node not in node_condition_counts:
            node_condition_counts[node] = {'target_count': 0, 'total_count': 0}
        node_condition_counts[node]['total_count'] += 1
        if condition == target_condition:
            node_condition_counts[node]['target_count'] += 1

node_proportions = {
    node: (counts['target_count'] / counts['total_count']) if counts['total_count'] > 0 else 0
    for node, counts in node_condition_counts.items()
}

def get_color(proportion):
    r = int(255 - (255 - 50) * proportion)    #LL 222        #PD 50    #cva 225   #HC 192 
    g = int(255 - (255 - 224) * proportion)   #LL 225       #PD 224    #cva 83   #HC 143
    b = int(255 - (255 - 197) * proportion)   #LL 94       #PD 197    #cva 80   #HC 219
    return f"rgb({r}, {g}, {b})"

# def get_color(proportion):   #For Red or Blue
#     intensity = int(255 * proportion) 
#     # return f"rgb(255, {255 - intensity}, {255 - intensity})"  # For Red
#     return f"rgb({255 - intensity}, {255 - intensity}, 255)"  # For Blue

# def get_color(proportion):   #For Green
#     red_blue_intensity = int(255 * (1 - proportion))  # Scale from 255 (white) to 0
#     green_intensity = int(127 * proportion)  # Scale from 0 to 127 (darker green)
#     return f"rgb({red_blue_intensity}, {green_intensity + red_blue_intensity}, {red_blue_intensity})"

# def get_color(proportion):   #For purple
#     red = int(255 - (155 * proportion))  # From 255 to 100
#     green = int(255 - (255 * proportion))  # From 255 to 0
#     blue = int(255 - (105 * proportion))  # From 255 to 150
#     return f"rgb({red}, {green}, {blue})"



node_colors = {node: get_color(proportion) for node, proportion in node_proportions.items()}

# ---------------- PLOTLY PLOTTING CODE ---------------- #
node_x = []
node_y = []
node_marker_colors = []
for node in G.nodes():
    if node in node_positions:
        x, y = node_positions[node]['x'], node_positions[node]['y']
        node_x.append(x)
        node_y.append(y)
        node_number = node.split('_')[0].replace("cube", "")  # Extract number only
        node_marker_colors.append(node_colors.get(node_number, "rgb(0,0,0)"))


node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(size=8, color=node_marker_colors, line=dict(width=0.3, color='grey')),
    hoverinfo='none', 
)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.3, color='grey'),
    hoverinfo='none',
    mode='lines'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    autosize=False,
    width=300,
    height=300,
    showlegend=False,
    hovermode='closest',
    margin=dict(b=0, l=0, r=0, t=0),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='white',
    paper_bgcolor='white'
)

fig.show()

## Save the mapper

In [ ]:
pio.write_image(fig, "")

## PostHoc 3. Mapper Coloration Based on Outcomes (heatmap)

<br> Please add your outcome columns to 'df2' </br>
<br> Also Please update the config location </br>
<br> You can change the weighting and normalization techniques in the code </br>

In [ ]:

# Load node positions from config file
with open("", "r") as file:
    node_positions = json.load(file)


target_conditions = ["LL"]  # List of conditions to merge
gradient_marker = "AMPPRO"  # Feature for color gradient
# "x10MWT_SLOW_m_s_"       "x10MWT_SSV_m_s_"       "x10MWT_FAST_m_s_"
# "TREAD_SPEED_mph_"        "x6MWT_m_"
# "TUG_s_"                  "TUG_DUALTASK_s_"      
# "FGA"                     "FMA_UE"     
# Step 1: Identify which nodes contain the target conditions and store severity scores
node_severity_scores = {}

for _, row in df2.iterrows():
    nodes = row["node_numbers"]
    condition = row["Condition"]
    severity_value = row[gradient_marker]  # Extract severity value
    
    if isinstance(nodes, str):
        nodes = ast.literal_eval(nodes)  
    
    for node in nodes:
        node_str = str(node)  # Ensure consistent formatting
        if node_str not in node_severity_scores:
            node_severity_scores[node_str] = {'weighted_value': 0, 'total_strides': 0, 'total_target_strides': 0}

        node_severity_scores[node_str]['total_strides'] += 1  # Count total appearances
        
        # If the sample belongs to any of the target conditions, treat them as one
        if condition in target_conditions:
            node_severity_scores[node_str]['weighted_value'] += severity_value  # Sum severity for target conditions
            node_severity_scores[node_str]['total_target_strides'] += 1  # Count only target samples

# ---------------------------------------
# 🔹 Choose Weighting Method (Uncomment to Switch)
# ---------------------------------------
# node_weighted_scores = {
#     node: (data['weighted_value'] / data['total_strides']) if data['total_strides'] > 0 else 0
#     for node, data in node_severity_scores.items()
# }

# Alternative (Only Average Across Target Condition Samples)
node_weighted_scores = {
    node: (data['weighted_value'] / data['total_target_strides']) if data['total_target_strides'] > 0 else 0
    for node, data in node_severity_scores.items()
}

# Step 2: Normalize Severity Using Percentile-Based Scaling
# max_severity = max(node_weighted_scores.values(), default=1)

# if max_severity > 0:
#     values = np.array(list(node_weighted_scores.values()))
#     lower, upper = np.percentile(values, [5, 95])  # Clip extreme values
#     scale_range = upper - lower if upper - lower != 0 else 1  # Prevent division by zero
#     normalized_severity = {
#         node: max(0, min(1, (score - lower) / scale_range)) for node, score in node_weighted_scores.items()
#     }
# else:
#     normalized_severity = {node: 0 for node in node_weighted_scores}



# # old normalization method
# values = np.array(list(node_weighted_scores.values()))
# lower, upper = np.percentile(values, [5, 95])
# scale_range = upper - lower if upper - lower != 0 else 1

# # Don’t clip to upper during normalization — instead, map values > 95% to 1.0 but preserve rank
# normalized_severity = {
#     node: max(0, min(1, (score - lower) / scale_range))
#     for node, score in node_weighted_scores.items()
# }

#new normalization method
values = np.array(list(node_weighted_scores.values()))
vmin = values.min()
vmax = values.max()
scale_range = vmax - vmin if vmax != vmin else 1

normalized_severity = {
    node: (score - vmin) / scale_range
    for node, score in node_weighted_scores.items()
}




# min_val = min(values)
# max_val = max(values)
# scale_range = max_val - min_val if max_val - min_val != 0 else 1

# normalized_severity = {
#     node: (score - min_val) / scale_range
#     for node, score in node_weighted_scores.items()
# }



# Step 3: Use High-Contrast Colormap for Severity Mapping
cmap = cm.get_cmap("inferno_r")  # Use a high-contrast colormap (Change to "inferno", "viridis", etc.)


def get_colormap_color(proportion):
    proportion = min(proportion, 0.95)  # prevents mapping to weird yellow at 1.0
    rgba = cmap(proportion)
    rgb = tuple(int(255 * c) for c in rgba[:3])
    return f"rgb{rgb}"


node_colors = {
    f"cube{node}_cluster0": (
        get_colormap_color(normalized_severity.get(node, 0)) if node in normalized_severity and node_severity_scores[node]["total_target_strides"] > 0
        else "rgb(255, 255, 255)"  # Keep nodes white if no target condition
    )
    for node in node_weighted_scores
}


# Step 4: Extract all nodes and edges
G = nx.Graph()
G.add_nodes_from(dfmapper["Nodes"])

edges_added = set()
for _, row in dfmapper.iterrows():
    node = row["Nodes"]
    edges = ast.literal_eval(row["Edges"]) if isinstance(row["Edges"], str) else row["Edges"]
    
    for neighbor in edges:
        if (neighbor, node) not in edges_added and (node, neighbor) not in edges_added:
            G.add_edge(node, neighbor)
            edges_added.add((node, neighbor))

# Step 5: Extract node positions and prepare for plotting
node_x = []
node_y = []
node_marker_colors = []

for node in G.nodes():
    if node in node_positions:
        x, y = node_positions[node]["x"], node_positions[node]["y"]
        node_x.append(x)
        node_y.append(y)
        
        node_number = node.split('_')[0].replace("cube", "")  # Extract node number
        node_marker_colors.append(node_colors.get(node, "rgb(255, 255, 255)"))  # Default to White

# Step 6: Extract edges
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = node_positions[edge[0]]["x"], node_positions[edge[0]]["y"]
    x1, y1 = node_positions[edge[1]]["x"], node_positions[edge[1]]["y"]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

# Step 7: Create Plotly traces
node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    marker=dict(size=8, color=node_marker_colors, line=dict(width=0.4, color='grey')),
    hoverinfo='none',
)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.3, color='grey'),
    hoverinfo='none',
    mode='lines'
)

# Step 8: Create and show figure
fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    autosize=False,
    width=300,
    height=300,
    showlegend=False,
    hovermode='closest',
    margin=dict(b=0, l=0, r=0, t=0),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig.show()

## Save the Mapper as SVG

In [ ]:
pio.write_image(fig, "")

## 8. IF you need PCA 

If you need to combine multiple outcomes and present them as PCA use this and update PostHoc 3 code.

In [ ]:

filtered_df = df2[df2['Condition'].isin(['HC'])]
X = filtered_df[[ "x10MWT_SLOW_m_s_","x10MWT_SSV_m_s_","x10MWT_FAST_m_s_","x6MWT_m_"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=1)
pc1 = pca.fit_transform(X_scaled)
print(pca.components_)
filtered_df["PC1"] = pc1
df2.loc[df2['Condition'].isin(['HC']), 'PC1'] = filtered_df['PC1']
df2
